In [ ]:
# 04_multimodal_dataset.ipynb
#  =================================================
# PURPOSE: Define PyTorch Dataset classes for:
#   - Multimodal (image + clinical)
#   - Image-only ablation
#   - Clinical-only ablation
#   - Vision-Language Model (VLM) fine-tuning with synthesized text prompts
#  =================================================

# --- CELL 1: Setup ---
from __future__ import annotations
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

OUTPUT_DIR = Path("./data/processed")
VLM_CACHE = OUTPUT_DIR / "vlm_cache"
VLM_CACHE.mkdir(parents=True, exist_ok=True)

IMG_SIZE = 224
torch.manual_seed(42)
np.random.seed(42)

print(f"VLM cache: {VLM_CACHE}")
print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")


VLM cache: data\processed\vlm_cache
PyTorch  : 2.11.0+cu128
CUDA     : True


In [14]:

# --- CELL 2: Load Metadata ---
clinical_df = pd.read_csv(OUTPUT_DIR / "clinical_features_merged.csv")
frames_df = pd.read_csv(OUTPUT_DIR / "frames_index.csv")
diagnosis_df = pd.read_csv(OUTPUT_DIR / "diagnosis_labels.csv")

frames_df["ID"] = frames_df["ID"].astype(str)
diagnosis_df["ID"] = diagnosis_df["ID"].astype(int)

non_numeric = clinical_df.select_dtypes(exclude="number").columns.tolist()
assert not non_numeric, f"Non-numeric columns present: {non_numeric}"

print(f"Clinical rows  : {len(clinical_df)}")
print(f"Frame records  : {len(frames_df)}")
print(f"Patients w/ frames: {frames_df['ID'].nunique()}")

Clinical rows  : 377
Frame records  : 13353
Patients w/ frames: 376


In [15]:
# --- CELL 3: Transforms ---
train_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

eval_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

In [16]:
# --- CELL 4: Dataset Classes ---
EXCLUDE_COLS = {"ID", "has_cad", "max_stenosis", "num_diseased_vessels"}

def feature_columns(clinical_df: pd.DataFrame) -> list[str]:
    return [c for c in clinical_df.columns if c not in EXCLUDE_COLS]


class BaseAngioDataset(Dataset):
    """Base class with shared image loading logic."""
    def __init__(self, patient_df, frames_df=None, clinical_df=None,
                 transform=None, aggregate="mean"):
        self.patients = patient_df.reset_index(drop=True)
        self.frames = frames_df
        self.clinical = clinical_df.set_index("ID") if clinical_df is not None else None
        self.transform = transform
        self.aggregate = aggregate
        if self.clinical is not None:
            self.feature_cols = feature_columns(clinical_df)

    def __len__(self) -> int:
        return len(self.patients)

    def _load_frames(self, pid: int) -> list[torch.Tensor]:
        subset = self.frames[self.frames["ID"] == str(int(pid))]
        tensors = []
        for path in subset["path"].tolist():
            try:
                img = Image.open(path).convert("RGB")
                tensors.append(self.transform(img) if self.transform
                               else transforms.ToTensor()(img))
            except Exception:
                continue
        return tensors

    def _aggregate(self, tensors: list[torch.Tensor]) -> torch.Tensor:
        if not tensors:
            return torch.zeros(3, IMG_SIZE, IMG_SIZE)
        stack = torch.stack(tensors)
        if self.aggregate == "max":   return stack.max(dim=0).values
        if self.aggregate == "first": return stack[0]
        return stack.mean(dim=0)


class AngioCADDataset(BaseAngioDataset):
    """Image + clinical table for supervised multimodal training."""
    def __getitem__(self, idx: int):
        row = self.patients.iloc[idx]
        pid = int(row["ID"])
        img = self._aggregate(self._load_frames(pid))
        clinical = self.clinical.loc[pid, self.feature_cols].values.astype(np.float32)
        label = torch.tensor(int(row["has_cad"]), dtype=torch.long)
        return img, torch.from_numpy(clinical), label


class ImageOnlyDataset(BaseAngioDataset):
    """Image-only ablation."""
    def __getitem__(self, idx: int):
        row = self.patients.iloc[idx]
        img = self._aggregate(self._load_frames(int(row["ID"])))
        label = torch.tensor(int(row["has_cad"]), dtype=torch.long)
        return img, label


class ClinicalOnlyDataset(BaseAngioDataset):
    """Clinical-only ablation."""
    def __init__(self, patient_df, clinical_df):
        super().__init__(patient_df, frames_df=None, clinical_df=clinical_df)

    def __getitem__(self, idx: int):
        row = self.patients.iloc[idx]
        pid = int(row["ID"])
        clinical = self.clinical.loc[pid, self.feature_cols].values.astype(np.float32)
        label = torch.tensor(int(row["has_cad"]), dtype=torch.long)
        return torch.from_numpy(clinical), label


class VLMDataset(BaseAngioDataset):
    """Image + natural-language prompt for vision-language fine-tuning.

    Text prompts are synthesized deterministically from tabular metadata.
    This provides a real text modality for CLIP/BiomedCLIP-style training
    without requiring external clinical notes.
    """
    PROMPT_TEMPLATE = (
        "A coronary angiography of a {age}-year-old {sex} patient. "
        "Diabetes: {dm}. Hypertension: {htn}. Smoker: {smoker}. "
        "Hyperlipidemia: {hlp}. LVEF: {lvef} percent. "
        "The finding is {label}."
    )

    def __init__(self, patient_df, frames_df, original_df, transform=None, aggregate="mean"):
        super().__init__(patient_df, frames_df, clinical_df=None, transform=transform, aggregate=aggregate)
        self.original = original_df.set_index("ID")

    def _build_prompt(self, pid: int) -> str:
        row = self.original.loc[pid]
        age = int(row["Age"]) if not pd.isna(row["Age"]) else 60
        sex = str(row.get("Sex", "unknown")).lower()
        dm = "yes" if row.get("DM", 0) == 1 else "no"
        htn = "yes" if row.get("HTN", 0) == 1 else "no"
        smoker = "yes" if row.get("Current Smoker", 0) == 1 else "no"
        hlp = "yes" if row.get("Hyperlipidemia", 0) == 1 else "no"
        lvef = row.get("LVEF", None)
        lvef = int(lvef) if not pd.isna(lvef) else 55
        label_text = "significant stenosis" if int(row.get("has_cad", 0)) == 1 else "normal coronary arteries"
        return self.PROMPT_TEMPLATE.format(
            age=age, sex=sex, dm=dm, htn=htn, smoker=smoker,
            hlp=hlp, lvef=lvef, label=label_text,
        )

    def __getitem__(self, idx: int):
        row = self.patients.iloc[idx]
        pid = int(row["ID"])
        image = self._aggregate(self._load_frames(pid))
        prompt = self._build_prompt(pid)
        label = torch.tensor(int(row["has_cad"]), dtype=torch.long)
        return image, prompt, label


In [17]:
# --- CELL 5: Sanity Check ---
original_df = pd.read_csv(OUTPUT_DIR / "clinical_features_original.csv")

with open(OUTPUT_DIR / "splits.json") as f:
    splits = json.load(f)

train_df = clinical_df[clinical_df["ID"].isin(splits["train"])]
val_df = clinical_df[clinical_df["ID"].isin(splits["val"])]
test_df = clinical_df[clinical_df["ID"].isin(splits["test"])]

train_ds = AngioCADDataset(train_df, frames_df, clinical_df, transform=train_tfms)
val_ds = AngioCADDataset(val_df, frames_df, clinical_df, transform=eval_tfms)
test_ds = AngioCADDataset(test_df, frames_df, clinical_df, transform=eval_tfms)

img, clin, label = train_ds[0]
print(f"Image    : {img.shape}")
print(f"Clinical : {clin.shape}")
print(f"Label    : {label.item()}")
print(f"Splits   : train={len(train_ds)} val={len(val_ds)} test={len(test_ds)}")

vlm_ds = VLMDataset(train_df.head(3), frames_df, original_df, transform=eval_tfms)
_, prompt, _ = vlm_ds[0]
print(f"\nSample VLM prompt:\n  {prompt}")

Image    : torch.Size([3, 224, 224])
Clinical : torch.Size([55])
Label    : 1
Splits   : train=255 val=46 test=76

Sample VLM prompt:
  A coronary angiography of a 52-year-old male patient. Diabetes: no. Hypertension: no. Smoker: no. Hyperlipidemia: no. LVEF: 0 percent. The finding is significant stenosis.
